In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.model_selection import train_test_split

In [ ]:
image_dir = "/content/drive/MyDrive/deep_learning_lab/skin_image_dataset/image_dataset"
csv_path = "/content/drive/MyDrive/deep_learning_lab/skin_image_dataset/HAM10000_metadata.csv"

print("Total images:", len(os.listdir(image_dir)))

Total images: 9526


In [ ]:
df = pd.read_csv(csv_path)
df['image_id'] = df['image_id'].astype(str) + ".jpg"

existing = set(os.listdir(image_dir))
df = df[df['image_id'].isin(existing)].reset_index(drop=True)

print("Valid images:", len(df))

Valid images: 9485


In [ ]:
df['binary_label'] = df['dx'].apply(lambda x: 0 if x=='nv' else 1)

In [ ]:
normal_df = df[df.binary_label==0]
abnormal_df = df[df.binary_label==1]

min_count = min(len(normal_df),len(abnormal_df))

normal_bal = normal_df.sample(min_count, random_state=42)
abnormal_bal = abnormal_df.sample(min_count, random_state=42)

balanced_df = pd.concat([normal_bal, abnormal_bal])
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
train_df, val_df = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df['binary_label'],
    random_state=42
)

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

In [ ]:
class SkinDataset(Dataset):
    def __init__(self,df,img_dir,tf,label_col):
        self.df=df.reset_index(drop=True)
        self.dir=img_dir
        self.tf=tf
        self.label_col=label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self,i):
        img_path = os.path.join(self.dir,self.df.loc[i,'image_id'])
        img = Image.open(img_path).convert("RGB")
        img = self.tf(img)
        label = self.df.loc[i,self.label_col]
        return img,label

In [ ]:
train_loader = DataLoader(
    SkinDataset(train_df,image_dir,train_tf,'binary_label'),
    batch_size=32,
    shuffle=True
)

In [ ]:
# Create unified 7-class mapping
label_map = {
    'nv':0,       # Normal
    'mel':1,
    'bkl':2,
    'bcc':3,
    'akiec':4,
    'vasc':5,
    'df':6
}

df['final_label'] = df['dx'].map(label_map)

In [ ]:
normal_df = df[df.final_label==0]
abnormal_df = df[df.final_label!=0]

min_count = min(len(normal_df), len(abnormal_df))

normal_bal = normal_df.sample(min_count, random_state=42)
abnormal_bal = abnormal_df.sample(min_count, random_state=42)

balanced_df = pd.concat([normal_bal, abnormal_bal])
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
train_df, val_df = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df['final_label'],
    random_state=42
)

In [ ]:
class SkinDataset(Dataset):
    def __init__(self,df,img_dir,tf):
        self.df=df.reset_index(drop=True)
        self.dir=img_dir
        self.tf=tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self,i):
        img_path = os.path.join(self.dir,self.df.loc[i,'image_id'])
        img = Image.open(img_path).convert("RGB")
        img = self.tf(img)
        label = self.df.loc[i,'final_label']
        return img,label

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Then create model
model = timm.create_model('vit_tiny_patch16_224', pretrained=True)
model.head = nn.Linear(model.head.in_features, 7)

# Then move to device
model = model.to(device)

model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for e in range(6):

    model.train()
    correct=total=0

    for x,y in train_loader:
        x,y=x.to(device),y.to(device)

        optimizer.zero_grad()
        out=model(x)
        loss=criterion(out,y)

        loss.backward()
        optimizer.step()

        _,pred=torch.max(out,1)
        total+=y.size(0)
        correct+=(pred==y).sum().item()

    print("Epoch",e+1,"Accuracy:",correct/total)

Epoch 1 Accuracy: 0.8150767987065481
Epoch 2 Accuracy: 0.8763136620856912
Epoch 3 Accuracy: 0.9128940986257074
Epoch 4 Accuracy: 0.936944219886823
Epoch 5 Accuracy: 0.9486661277283751
Epoch 6 Accuracy: 0.967663702506063


In [ ]:
torch.save(model.state_dict(),
           "/content/drive/MyDrive/deep_learning_lab/vit_7class_model.pth")

print("7-Class Model Saved Successfully")

7-Class Model Saved Successfully


In [ ]:
import torch
import torch.nn as nn
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate same architecture
model = timm.create_model('vit_tiny_patch16_224', pretrained=False)
model.head = nn.Linear(model.head.in_features, 7)
model = model.to(device)

# Load saved weights
model.load_state_dict(
    torch.load("/content/drive/MyDrive/deep_learning_lab/vit_7class_model.pth",
               map_location=device)
)

model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


In [1]:
from torchvision import transforms

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

In [ ]:
import torch
import torch.nn as nn
import timm
from PIL import Image
from torchvision import transforms
import os

# -----------------------------
# 1️⃣ Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# 2️⃣ Recreate Model Architecture
# -----------------------------
model = timm.create_model('vit_tiny_patch16_224', pretrained=False)
model.head = nn.Linear(model.head.in_features, 7)
model = model.to(device)

# -----------------------------
# 3️⃣ Load Saved Weights
# -----------------------------
model_path = "vit_7class_model.pth"

if not os.path.exists(model_path):
    print("Model file not found!")
    exit()

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("Model Loaded Successfully")

# -----------------------------
# 4️⃣ Transform
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

# -----------------------------
# 5️⃣ Image Input
# -----------------------------
image_path = input("Enter image filename: ")

if not os.path.exists(image_path):
    print("Image not found!")
    exit()

img = Image.open(image_path).convert("RGB")
input_tensor = transform(img).unsqueeze(0).to(device)

# -----------------------------
# 6️⃣ Prediction
# -----------------------------
with torch.no_grad():
    output = model(input_tensor)
    probs = torch.softmax(output,1)

classes = ['Normal','mel','bkl','bcc','akiec','vasc','df']

idx = torch.argmax(probs).item()
confidence = probs[0][idx].item()

print("\n===== RESULT =====")
print("Prediction :", classes[idx])
print("Confidence :", round(confidence*100,2), "%")

Using device: cpu
Model Loaded Successfully


IEEE PAPER COMPARISON